# InsPLAD Explainable QA Gold Builder

현재 `lvlm_insplad` 기준으로 QA를 자동 생성합니다.

- Main: 250 images → Q1 정상/이상 + Q2 defect type
- Localization subset: Main 중 `InsPLAD-seg`와 실제 매칭되는 이미지만 Q3 defect location 추가
- 결과는 `MyDrive/lvlm_insplad_qa/gold_v1/`에 저장
- 기존 5문항 smoke의 description/analysis는 원본 GT가 아니므로 gold에서 제외
- asset object bbox도 cropped fault image에서는 제외

Colab Secret:
- `GITHUB_TOKEN`: private repo clone용
- `ROBOFLOW_API_KEY`: Q3 localization subset 자동 다운로드용 (없어도 Main은 생성됨)

In [ ]:
# 0) 설정
MASTER_IMAGE_COUNT = 250
SEED = 42
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DRIVE_DATA_DIR = None  # 예: "datasets/InsPLAD"
OUTPUT_FOLDER = "lvlm_insplad_qa/gold_v1"

ROBOFLOW_WORKSPACE = "andreluizbvs"
ROBOFLOW_PROJECT = "insplad-seg"
ROBOFLOW_VERSION = 7
ROBOFLOW_FORMAT = "coco-segmentation"

# True면 기존 8,990 overlap을 SHA256으로 다시 확인. 기본은 이미 repo에서 검증한 결과 재사용.
STRICT_REHASH = False

EXPECTED_ZIP_SIZES = {
    "unsupervised_anomaly_detection.zip": 1_170_967_719,
    "supervised_fault_classification.zip": 874_753_167,
}

In [ ]:
# 1) Drive mount + 최신 repo clone
import base64, csv, io, json, os, random, shutil, subprocess, sys, zipfile
from pathlib import Path

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
OUT_ROOT = DRIVE_ROOT / OUTPUT_FOLDER
OUT_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
WORK_ROOT = Path("/content/insplad_qa_build")
for p in [REPO_DIR, WORK_ROOT]:
    if p.exists():
        shutil.rmtree(p)
WORK_ROOT.mkdir(parents=True)

def run(cmd, cwd=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:5]) + (" ..." if len(cmd) > 5 else ""))
    subprocess.run(cmd, cwd=cwd, check=True)

token = None
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

clone = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    clone += ["-c", f"http.extraHeader=AUTHORIZATION: basic {auth}"]
clone += ["clone", "--depth", "1", "--branch", REPO_REF,
          f"https://github.com/{REPO}.git", str(REPO_DIR)]
run(clone)

print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("output:", OUT_ROOT)

In [ ]:
# 2) Drive의 두 source ZIP 확인
def find_unique_file(filename):
    base = DRIVE_ROOT / DRIVE_DATA_DIR if DRIVE_DATA_DIR else DRIVE_ROOT
    direct = base / filename
    if direct.is_file():
        return direct
    matches = list(base.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} 을 Drive에서 찾지 못했습니다.")
    if len(matches) > 1:
        print("[주의] 여러 후보 중 첫 파일 사용:", matches[0])
    return matches[0]

UNSUP_ZIP = find_unique_file("unsupervised_anomaly_detection.zip")
SUP_ZIP = find_unique_file("supervised_fault_classification.zip")

for p in [UNSUP_ZIP, SUP_ZIP]:
    actual = p.stat().st_size
    expected = EXPECTED_ZIP_SIZES[p.name]
    if actual != expected:
        raise RuntimeError(f"{p.name} 크기 불일치: expected={expected}, actual={actual}")
    print("PASS", p.name, actual)

In [ ]:
# 3) 기존 repo 로직으로 verified Aligned Core 1,378 재생성
MAN = WORK_ROOT / "manifests"
BENCH = WORK_ROOT / "benchmark"
MAN.mkdir(parents=True)
BENCH.mkdir(parents=True)

unsup_manifest = MAN / "unsup.csv"
sup_manifest = MAN / "sup.csv"
sup_preview = MAN / "sup_preview.json"

run([sys.executable, REPO_DIR/"scripts/inspect_zip_manifest.py", UNSUP_ZIP,
     "--out-csv", unsup_manifest, "--out-json", MAN/"unsup.json"])
run([sys.executable, REPO_DIR/"scripts/inspect_zip_manifest.py", SUP_ZIP,
     "--out-csv", sup_manifest, "--out-json", MAN/"sup.json",
     "--text-preview-json", sup_preview])

overlap_summary = WORK_ROOT / "image_overlap_summary.json"
if STRICT_REHASH:
    run([sys.executable, REPO_DIR/"scripts/hash_overlap.py",
         "--anomaly-zip", UNSUP_ZIP, "--fault-zip", SUP_ZIP,
         "--anomaly-manifest", unsup_manifest, "--fault-manifest", sup_manifest,
         "--out", WORK_ROOT/"image_overlap.csv", "--summary", overlap_summary])
else:
    shutil.copy2(REPO_DIR/"reports/image_overlap_summary.json", overlap_summary)

run([sys.executable, REPO_DIR/"scripts/build_benchmark.py",
     "--anomaly-manifest", unsup_manifest,
     "--fault-manifest", sup_manifest,
     "--overlap-summary", overlap_summary,
     "--annotation-previews", sup_preview,
     "--repo-root", BENCH,
     "--seed", str(SEED)])

ALIGNED_CSV = BENCH / "reports/aligned_core.csv"
with ALIGNED_CSV.open(encoding="utf-8-sig", newline="") as f:
    aligned = list(csv.DictReader(f))
assert len(aligned) == 1378, len(aligned)
print("Aligned Core:", len(aligned))

In [ ]:
# 4) 공식 InsPLAD-seg v7 준비
SEG_ROOT = OUT_ROOT / "segmentation_source" / f"insplad-seg-v{ROBOFLOW_VERSION}"
coco_files = list(SEG_ROOT.rglob("_annotations.coco.json")) if SEG_ROOT.exists() else []

if not coco_files:
    rf_key = None
    try:
        rf_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        pass

    if rf_key:
        run([sys.executable, "-m", "pip", "install", "-q", "roboflow"])
        from roboflow import Roboflow
        rf = Roboflow(api_key=rf_key)
        project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        project.version(ROBOFLOW_VERSION).download(
            ROBOFLOW_FORMAT, location=str(SEG_ROOT), overwrite=False
        )
        coco_files = list(SEG_ROOT.rglob("_annotations.coco.json"))
    else:
        print("ROBOFLOW_API_KEY 없음 → Main QA만 생성합니다.")

SEG_AVAILABLE = bool(coco_files)
print("segmentation available:", SEG_AVAILABLE)
if SEG_AVAILABLE:
    print("COCO:", *coco_files, sep="\n- ")

In [ ]:
# 5) QA 생성 → Drive 저장
cmd = [
    sys.executable, REPO_DIR/"scripts/build_explainable_qa_gold.py",
    "--aligned-core-csv", ALIGNED_CSV,
    "--out-dir", OUT_ROOT,
    "--master-count", str(MASTER_IMAGE_COUNT),
    "--seed", str(SEED),
    "--seg-version", str(ROBOFLOW_VERSION),
]
if SEG_AVAILABLE:
    cmd += ["--seg-root", SEG_ROOT]
run(cmd)

summary = json.loads((OUT_ROOT/"build_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\n생성 파일:")
for p in sorted(OUT_ROOT.iterdir()):
    if p.is_file():
        print("-", p.name)

In [ ]:
# 6) 분포 확인
import pandas as pd
counts = pd.read_csv(OUT_ROOT/"master_counts.csv")
display(counts)

print("총 Main images:", counts["total"].sum())
print("총 anomaly:", counts["anomalous"].sum())
print("총 normal:", counts["normal"].sum())

audit_path = OUT_ROOT/"localization_match_audit.csv"
if audit_path.is_file() and audit_path.stat().st_size:
    audit = pd.read_csv(audit_path)
    print("\nLocalization match status:")
    display(audit["status"].value_counts().rename_axis("status").reset_index(name="count"))

In [ ]:
# 7) 이미지 + GT 빠른 검수
# Main에서 랜덤 이미지를 보여주고, localization GT가 있으면 bbox를 같이 그립니다.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

master_file = next(OUT_ROOT.glob("master_images_*.jsonl"))
master = read_jsonl(master_file)
loc_file = OUT_ROOT/"localization_images.jsonl"
localization = read_jsonl(loc_file) if loc_file.is_file() else []
loc_by_id = {x["core_id"]: x for x in localization}

rng = random.Random(SEED)
preview = rng.sample(master, min(12, len(master)))

with zipfile.ZipFile(SUP_ZIP) as zf:
    for row in preview:
        with zf.open(row["image"]) as f:
            image = Image.open(io.BytesIO(f.read())).convert("RGB")
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(image)
        ax.set_title(f"{row['asset_category']} | {row['anomaly_gt']} | {row['defect_gt']}")
        ax.axis("off")

        loc = loc_by_id.get(row["core_id"])
        if loc:
            w, h = image.size
            for x, y, bw, bh in loc["localization_gt"]["normalized_bboxes_xywh"]:
                ax.add_patch(Rectangle((x*w, y*h), bw*w, bh*h, fill=False, linewidth=2))
            print(row["core_id"], "Q3 GT:", loc["localization_gt"]["coarse_region"])
        plt.show()

print("검수표:", OUT_ROOT/"manual_review.csv")

## 결과 해석

Drive의 `gold_v1` 폴더에서 보면 됩니다.

- `master_images_250.jsonl`: 선택된 250장과 Q1/Q2 GT
- `master_qa_500.jsonl`: Main MCQA
- `localization_images.jsonl`: 실제 segmentation GT와 매칭된 Main subset
- `localization_qa_<N>.jsonl`: Q1/Q2/Q3를 모두 가진 QA
- `localization_match_audit.csv`: 왜 매칭/제외됐는지
- `manual_review.csv`: 사람이 최종 검수할 표
- `DATASET_CARD.md`, `build_summary.json`: 논문/실험 기록용

Localization 개수는 notebook 실행 시 실제 filename/category GT를 맞춘 뒤 확정됩니다.